In [1]:
!pip install requests
!pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


5 grandes ligas

In [2]:
import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

BASE_URL = "https://www.fotmob.com/api/data/leagues"

ligas = {
    "Premier League": {"id": 47, "pais": "ENG"},
    "LaLiga": {"id": 87, "pais": "ESP"},
    "Serie A": {"id": 55, "pais": "ITA"},
    "Bundesliga": {"id": 54, "pais": "GER"},
    "Ligue 1": {"id": 53, "pais": "FRA"},
}

# Nombres de columna de la tabla xG tal como los da fotmob -> nombres "bonitos"
RENOMBRE_XG = {
    "name": "team",
    "played": "J",
    "xg": "xG",
    "xgConceded": "xGA",
    "xPoints": "xPTS",
    "xgDiff": "xG_diff",
    "xgConcededDiff": "xGA_diff",
    "xPointsDiff": "xPTS_diff",
    "position": "pos_real",
    "xPosition": "pos_xG",
    "xPositionDiff": "pos_diff",
}

# dejamos teamId dentro para poder construir el logo; se puede quitar después si molesta
COLUMNAS_XG_UTILES = ["teamId", "team", "J", "xG", "xG_diff", "xGA", "xGA_diff", "xPTS", "xPTS_diff",
                      "pos_real", "pos_xG", "pos_diff"]


def _url_logo(team_id):
    if pd.isna(team_id):
        return None
    return f"https://images.fotmob.com/image_resources/logo/teamlogo/{int(team_id)}.png"


def anadir_logos(df):
    """Añade la columna 'logo' usando el id del equipo (id o teamId)."""
    df = df.copy()
    id_col = "id" if "id" in df.columns else ("teamId" if "teamId" in df.columns else None)
    if id_col is not None:
        df["logo"] = df[id_col].apply(_url_logo)
    return df


def extraer_tablas(data):
    """
    Todas las tablas (all/home/away/form/xg) viven en:
    data['table'][*]['data']['table'] = {"all":[...], "home":[...],
                                          "away":[...], "form":[...], "xg":[...]}
    """
    tablas = data.get("table", [])
    resultados = {"all": None, "home": None, "away": None, "last5": None, "xG": None}
    mapeo_tipos = {"all": "all", "home": "home", "away": "away", "form": "last5", "xg": "xG"}

    for bloque in tablas:
        tabla_obj = bloque.get("data", {}).get("table", {})
        if not isinstance(tabla_obj, dict):
            continue

        for clave_json, clave_salida in mapeo_tipos.items():
            if clave_json in tabla_obj and tabla_obj[clave_json]:
                df = pd.json_normalize(tabla_obj[clave_json], sep="_")

                # por si el equipo viniera anidado como dict en vez de columnas planas
                if "team" in df.columns and df["team"].apply(lambda t: isinstance(t, dict)).any():
                    df["logoUrl"] = df["team"].apply(lambda t: t.get("logo") if isinstance(t, dict) else None)
                    df["name"] = df["team"].apply(lambda t: t.get("name") if isinstance(t, dict) else None)
                    df["shortName"] = df["team"].apply(lambda t: t.get("shortName") if isinstance(t, dict) else None)

                # tabla xG -> renombrar a nombres tipo web y quedarnos con lo útil
                if clave_json == "xg":
                    df = df.rename(columns=RENOMBRE_XG)
                    df = df[[c for c in COLUMNAS_XG_UTILES if c in df.columns]]
                    df = df.sort_values("xPTS", ascending=False).reset_index(drop=True)

                df = anadir_logos(df)
                resultados[clave_salida] = df

    return resultados


def preparar_tablas(tablas_dict, nombre_liga, temporada):
    preparados = {}
    for tipo, df in tablas_dict.items():
        if df is not None:
            df = df.copy()
            df["Liga"] = nombre_liga
            df["Tipo"] = tipo
            df["Temporada"] = temporada
            columnas = ["Liga", "Tipo", "Temporada"] + [c for c in df.columns if c not in ["Liga", "Tipo", "Temporada"]]
            df = df[columnas]
            preparados[tipo] = df
    return preparados


def obtener_clasificacion(id_liga, nombre_liga, season_in=2025, season_out=2026, pais="ESP", debug=False):
    temporada = f"{season_in}/{season_out}"
    params = {"id": id_liga, "ccode3": pais, "season": temporada}
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    data = r.json()

    tablas = extraer_tablas(data)

    if debug:
        for tipo, df in tablas.items():
            print(f"  -> {tipo}: {'OK, columnas=' + str(list(df.columns)) if df is not None else 'NO ENCONTRADA'}")

    return preparar_tablas(tablas, nombre_liga, temporada)

In [3]:
clasificaciones = {}
season_in = 2025
season_out = 2026

for nombre_liga, info in ligas.items():
    print(f"\n🏆 {nombre_liga}")
    tablas = obtener_clasificacion(info["id"], nombre_liga, season_in, season_out, pais=info["pais"], debug=True)
    clasificaciones[nombre_liga] = tablas

    for tipo, df in tablas.items():
        print(f"\n📌 {nombre_liga} — {tipo.upper()}")
        if df is not None:
            display(df.head())
        else:
            print("No disponible")


🏆 Premier League
  -> all: OK, columnas=['name', 'shortName', 'id', 'pageUrl', 'deduction', 'ongoing', 'played', 'wins', 'draws', 'losses', 'scoresStr', 'goalConDiff', 'pts', 'idx', 'qualColor', 'logo']
  -> home: OK, columnas=['name', 'shortName', 'id', 'pageUrl', 'deduction', 'ongoing', 'played', 'wins', 'draws', 'losses', 'scoresStr', 'goalConDiff', 'goalsScored', 'pts', 'idx', 'qualColor', 'logo']
  -> away: OK, columnas=['name', 'shortName', 'id', 'pageUrl', 'deduction', 'ongoing', 'played', 'wins', 'draws', 'losses', 'scoresStr', 'goalConDiff', 'goalsScored', 'pts', 'idx', 'qualColor', 'logo']
  -> last5: OK, columnas=['name', 'shortName', 'featuredInMatch', 'id', 'pageUrl', 'played', 'wins', 'draws', 'losses', 'scoresStr', 'goalConDiff', 'goalsScored', 'pts', 'ongoing', 'idx', 'qualColor', 'logo']
  -> xG: OK, columnas=['teamId', 'team', 'J', 'xG', 'xG_diff', 'xGA', 'xGA_diff', 'xPTS', 'xPTS_diff', 'pos_real', 'pos_xG', 'pos_diff', 'logo']

📌 Premier League — ALL


,Liga,Tipo,Temporada,name,shortName,id,pageUrl,deduction,ongoing,played,wins,draws,losses,scoresStr,goalConDiff,pts,idx,qualColor,logo
0,Premier League,all,2025/2026,Arsenal,Arsenal,9825,/teams/9825/overview/arsenal,None,None,38,26,7,5,71-27,44,85,1,#2AD572,https://images.fotmob.com/image_resources/logo...
1,Premier League,all,2025/2026,Manchester City,Man City,8456,/teams/8456/overview/manchester-city,None,None,38,23,9,6,77-35,42,78,2,#2AD572,https://images.fotmob.com/image_resources/logo...
2,Premier League,all,2025/2026,Manchester United,Man United,10260,/teams/10260/overview/manchester-united,None,None,38,20,11,7,69-50,19,71,3,#2AD572,https://images.fotmob.com/image_resources/logo...
3,Premier League,all,2025/2026,Aston Villa,Aston Villa,10252,/teams/10252/overview/aston-villa,None,None,38,19,8,11,56-49,7,65,4,#2AD572,https://images.fotmob.com/image_resources/logo...
4,Premier League,all,2025/2026,Liverpool,Liverpool,8650,/teams/8650/overview/liverpool,None,None,38,17,9,12,63-53,10,60,5,#2AD572,https://images.fotmob.com/image_resources/logo...



📌 Premier League — HOME


,Liga,Tipo,Temporada,name,shortName,id,pageUrl,deduction,ongoing,played,wins,draws,losses,scoresStr,goalConDiff,goalsScored,pts,idx,qualColor,logo
0,Premier League,home,2025/2026,Arsenal,Arsenal,9825,/teams/9825/overview/arsenal,None,None,19,15,2,2,41-11,30,41,47,1,#2AD572,https://images.fotmob.com/image_resources/logo...
1,Premier League,home,2025/2026,Manchester City,Man City,8456,/teams/8456/overview/manchester-city,None,None,19,14,3,2,45-14,31,45,45,2,#2AD572,https://images.fotmob.com/image_resources/logo...
2,Premier League,home,2025/2026,Manchester United,Man United,10260,/teams/10260/overview/manchester-united,None,None,19,13,3,3,39-24,15,39,42,3,#2AD572,https://images.fotmob.com/image_resources/logo...
3,Premier League,home,2025/2026,Aston Villa,Aston Villa,10252,/teams/10252/overview/aston-villa,None,None,19,12,2,5,32-22,10,32,38,4,#2AD572,https://images.fotmob.com/image_resources/logo...
4,Premier League,home,2025/2026,Liverpool,Liverpool,8650,/teams/8650/overview/liverpool,None,None,19,10,6,3,34-20,14,34,36,5,#2AD572,https://images.fotmob.com/image_resources/logo...



📌 Premier League — AWAY


...

,Liga,Tipo,Temporada,name,shortName,id,pageUrl,deduction,ongoing,played,wins,draws,losses,scoresStr,goalConDiff,goalsScored,pts,idx,qualColor,logo
0,Ligue 1,away,2025/2026,Paris Saint-Germain,PSG,9847,/teams/9847/overview/paris-saint-germain,None,None,17,11,2,4,33-17,16,33,35,1,#2AD572,https://images.fotmob.com/image_resources/logo...
1,Ligue 1,away,2025/2026,Lille,Lille,8639,/teams/8639/overview/lille,None,None,17,10,2,5,28-20,8,28,32,2,#2AD572,https://images.fotmob.com/image_resources/logo...
2,Ligue 1,away,2025/2026,Lens,Lens,8588,/teams/8588/overview/lens,None,None,17,8,4,5,31-22,9,31,28,3,#2AD572,https://images.fotmob.com/image_resources/logo...
3,Ligue 1,away,2025/2026,Rennes,Rennes,9851,/teams/9851/overview/rennes,None,None,17,7,4,6,29-33,-4,29,25,4,#FFD908,https://images.fotmob.com/image_resources/logo...
4,Ligue 1,away,2025/2026,Lyon,Lyon,9748,/teams/9748/overview/lyon,None,None,17,6,5,6,23-22,1,23,23,5,#0046A7,https://images.fotmob.com/image_resources/logo...



📌 Ligue 1 — LAST5


,Liga,Tipo,Temporada,name,shortName,featuredInMatch,id,pageUrl,played,wins,draws,losses,scoresStr,goalConDiff,goalsScored,pts,ongoing,idx,qualColor,logo
0,Ligue 1,last5,2025/2026,Paris Saint-Germain,PSG,False,9847,/teams/9847/overview/paris-saint-germain,5,3,1,1,9-4,5,9,10,None,1,#2AD572,https://images.fotmob.com/image_resources/logo...
1,Ligue 1,last5,2025/2026,Auxerre,Auxerre,False,8583,/teams/8583/overview/auxerre,5,3,1,1,11-7,4,11,10,None,2,#2AD572,https://images.fotmob.com/image_resources/logo...
2,Ligue 1,last5,2025/2026,Strasbourg,Strasbourg,False,9848,/teams/9848/overview/strasbourg,5,3,1,1,12-10,2,12,10,None,3,#2AD572,https://images.fotmob.com/image_resources/logo...
3,Ligue 1,last5,2025/2026,Paris FC,Paris FC,False,6379,/teams/6379/overview/paris-fc,5,3,0,2,10-5,5,10,9,None,4,#FFD908,https://images.fotmob.com/image_resources/logo...
4,Ligue 1,last5,2025/2026,Rennes,Rennes,False,9851,/teams/9851/overview/rennes,5,3,0,2,10-9,1,10,9,None,5,#0046A7,https://images.fotmob.com/image_resources/logo...



📌 Ligue 1 — XG


,Liga,Tipo,Temporada,teamId,team,J,xG,xG_diff,xGA,xGA_diff,xPTS,xPTS_diff,pos_real,pos_xG,pos_diff,logo
0,Ligue 1,xG,2025/2026,9847,Paris Saint-Germain,34,72.0950,1.9050,31.9307,-2.9307,72.438917,3.561083,1,1,0,https://images.fotmob.com/image_resources/logo...
1,Ligue 1,xG,2025/2026,8588,Lens,34,67.6513,-1.6513,43.6623,-8.6623,62.150667,7.849333,2,2,0,https://images.fotmob.com/image_resources/logo...
2,Ligue 1,xG,2025/2026,8639,Lille,34,55.2639,-3.2639,36.8764,0.1236,58.951657,2.048343,3,3,0,https://images.fotmob.com/image_resources/logo...
3,Ligue 1,xG,2025/2026,8592,Marseille,34,61.9146,1.0854,47.8447,-2.8447,56.682383,2.317617,4,4,1,https://images.fotmob.com/image_resources/logo...
4,Ligue 1,xG,2025/2026,9748,Lyon,34,53.4969,-0.4969,45.4152,-5.4152,52.527850,7.472150,5,5,-1,https://images.fotmob.com/image_resources/logo...


In [4]:
# Comprobar que los logos cargan correctamente

clasificaciones["Premier League"]["xG"][["team", "logo", "xG", "xGA", "xPTS"]].head()

,team,logo,xG,xGA,xPTS
0,Arsenal,https://images.fotmob.com/image_resources/logo...,65.5997,28.3025,77.824531
1,Manchester City,https://images.fotmob.com/image_resources/logo...,71.3303,44.1955,71.038303
2,Manchester United,https://images.fotmob.com/image_resources/logo...,65.4876,48.5707,62.288436
3,Liverpool,https://images.fotmob.com/image_resources/logo...,61.3487,47.4282,61.609077
4,Chelsea,https://images.fotmob.com/image_resources/logo...,66.5331,52.2635,60.007062
